# Fine-Tuning Qwen 3.5 9B with QLoRA

This notebook demonstrates fine-tuning the Qwen 3.5 9B base model using QLoRA (Quantized Low-Rank Adaptation) on a custom criminal intelligence validation dataset.

## Requirements
- GPU: T4 (free tier) or better
- Runtime: GPU-enabled
- Dataset: train.jsonl (174 examples)

## What this notebook does:
1. Installs required dependencies
2. Loads the Qwen 3.5 9B model with 4-bit quantization
3. Prepares the dataset for training
4. Configures QLoRA parameters
5. Fine-tunes the model
6. Saves the trained adapter
7. Tests the model with sample inference

## Step 1: Install Dependencies

In [ ]:
# Install required packages
!pip install -q -U transformers==4.48.0
!pip install -q -U accelerate==1.2.1
!pip install -q -U peft==0.14.0
!pip install -q -U bitsandbytes==0.45.0
!pip install -q -U trl==0.12.2
!pip install -q -U datasets==3.2.0
!pip install -q -U sentencepiece==0.2.0

print("✅ All dependencies installed successfully!")

## Step 2: Import Libraries

In [ ]:
import os
import json
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline
)
from peft import (
    LoraConfig,
    PeftModel,
    prepare_model_for_kbit_training,
    get_peft_model
)
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Step 3: Upload Your Dataset

Upload your `train.jsonl` file using the file upload button on the left sidebar, or mount Google Drive if the file is there.

In [ ]:
# Option 1: Upload file directly
from google.colab import files

print("Please upload your train.jsonl file:")
uploaded = files.upload()

# Verify upload
if 'train.jsonl' in uploaded:
    print(f"✅ File uploaded: {len(uploaded['train.jsonl'])} bytes")
else:
    print("❌ train.jsonl not found. Please upload it.")

In [ ]:
# Option 2: Mount Google Drive (uncomment if using Drive)
# from google.colab import drive
# drive.mount('/content/drive')
# 
# # Update this path to your file location in Drive
# !cp '/content/drive/MyDrive/path/to/train.jsonl' '/content/train.jsonl'

## Step 4: Load and Prepare Dataset

In [ ]:
# Load JSONL dataset
def load_jsonl(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line.strip()))
    return data

# Load the dataset
train_data = load_jsonl('train.jsonl')
print(f"✅ Loaded {len(train_data)} training examples")

# Display sample
print("\n📋 Sample record:")
print(json.dumps(train_data[0], indent=2)[:500] + "...")

In [ ]:
# Format data for training (ChatML format for Qwen)
def format_chat_template(example):
    """Convert messages to Qwen's chat format"""
    messages = example['messages']
    
    # Format: <|im_start|>role\ncontent<|im_end|>
    formatted_text = ""
    for msg in messages:
        role = msg['role']
        content = msg['content']
        
        # Handle dict content (convert to JSON string)
        if isinstance(content, dict):
            content = json.dumps(content, ensure_ascii=False)
        
        formatted_text += f"<|im_start|>{role}\n{content}<|im_end|>\n"
    
    return {"text": formatted_text}

# Create Dataset object
dataset = Dataset.from_list(train_data)
dataset = dataset.map(format_chat_template, remove_columns=dataset.column_names)

print(f"✅ Dataset formatted: {len(dataset)} examples")
print("\n📝 Formatted example (first 500 chars):")
print(dataset[0]['text'][:500] + "...")

## Step 5: Configure Model and Tokenizer

In [ ]:
# Model configuration
model_name = "Qwen/Qwen2.5-9B"  # Base model
output_dir = "./qwen3.5-9b-qlora-finetuned"

# 4-bit quantization config for QLoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"📦 Loading model: {model_name}")
print("⚙️  Quantization: 4-bit NF4 with double quantization")

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True,
    use_fast=True
)

# Ensure pad token is set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"✅ Tokenizer loaded")
print(f"Vocab size: {len(tokenizer)}")
print(f"PAD token: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")
print(f"EOS token: {tokenizer.eos_token} (ID: {tokenizer.eos_token_id})")

In [ ]:
# Load model with quantization
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2" if torch.cuda.is_available() else "eager"
)

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

print("✅ Model loaded and prepared for training")
print(f"Model size: {model.get_memory_footprint() / 1e9:.2f} GB")

## Step 6: Configure LoRA Parameters

In [ ]:
# LoRA configuration
lora_config = LoraConfig(
    r=64,                           # LoRA rank
    lora_alpha=128,                 # LoRA alpha (scaling factor)
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],                              # Target all attention and MLP layers
    lora_dropout=0.05,              # Dropout for LoRA layers
    bias="none",
    task_type="CAUSAL_LM"
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)

# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
trainable_percentage = 100 * trainable_params / total_params

print(f"✅ LoRA configured")
print(f"Trainable params: {trainable_params:,} ({trainable_percentage:.4f}%)")
print(f"Total params: {total_params:,}")
print(f"LoRA rank: {lora_config.r}")
print(f"LoRA alpha: {lora_config.lora_alpha}")

## Step 7: Configure Training Arguments

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,                    # Number of epochs
    per_device_train_batch_size=1,         # Batch size per GPU
    gradient_accumulation_steps=8,         # Accumulate gradients over 8 steps
    gradient_checkpointing=True,           # Enable gradient checkpointing
    optim="paged_adamw_32bit",            # Optimizer
    learning_rate=2e-4,                    # Learning rate
    lr_scheduler_type="cosine",            # Learning rate schedule
    warmup_ratio=0.05,                     # Warmup steps
    logging_steps=10,                      # Log every 10 steps
    save_strategy="epoch",                 # Save checkpoint every epoch
    save_total_limit=2,                    # Keep only 2 checkpoints
    bf16=True,                             # Use bfloat16 precision
    tf32=True,                             # Use TF32 for faster training
    max_grad_norm=0.3,                     # Gradient clipping
    weight_decay=0.001,                    # Weight decay
    group_by_length=True,                  # Group sequences by length
    report_to="none",                      # Disable wandb/tensorboard
    seed=42,
)

print("✅ Training arguments configured")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Batch size: {training_args.per_device_train_batch_size}")
print(f"Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Learning rate: {training_args.learning_rate}")

## Step 8: Initialize Trainer

In [ ]:
# Initialize trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=2048,              # Maximum sequence length
    packing=False,                     # Don't pack multiple examples
)

print("✅ Trainer initialized")
print(f"Training samples: {len(trainer.train_dataset)}")
print(f"Total steps: {trainer.args.num_train_epochs * len(trainer.train_dataset) // (trainer.args.per_device_train_batch_size * trainer.args.gradient_accumulation_steps)}")

## Step 9: Start Training 🚀

In [ ]:
# Clear CUDA cache before training
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("🚀 Starting training...\n")
print("="*50)

# Train the model
trainer.train()

print("\n" + "="*50)
print("✅ Training completed!")

## Step 10: Save the Fine-Tuned Model

In [ ]:
# Save the LoRA adapter
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"✅ Model saved to: {output_dir}")
print(f"\nFiles saved:")
!ls -lh {output_dir}

## Step 11: Test the Fine-Tuned Model

In [ ]:
# Load the fine-tuned model for inference
print("Loading fine-tuned model for inference...")

# Reload base model
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)

# Load LoRA adapter
model_inference = PeftModel.from_pretrained(base_model, output_dir)

print("✅ Model loaded for inference")

In [ ]:
# Create test prompt
test_prompt = """<|im_start|>system
You are an evidence-constrained criminal intelligence validation model. Validate candidate intelligence only against the supplied evidence. Do not invent facts. Do not treat allegations, statements, inferences, or hypotheses as established facts. Do not merge entities solely because their names are similar. Return only valid JSON.<|im_end|>
<|im_start|>user
{
  "task": "VALIDATE_CANDIDATE",
  "case_context": {
    "case_id": "TEST-001/2024"
  },
  "evidence": [
    {
      "evidence_id": "EV-001",
      "document_id": "DOC-001",
      "section": "Police Report",
      "text": "On 15/01/2024 at 14:30 hrs, Officer Kumar observed suspect John Doe driving a red Honda Civic with license plate ABC-123."
    }
  ],
  "candidate": {
    "candidate_id": "CAND-001",
    "candidate_type": "RELATIONSHIP",
    "subject": {
      "entity_id": "ENT-001",
      "entity_type": "PERSON",
      "name": "John Doe"
    },
    "predicate": "DRIVER_OF",
    "object": {
      "entity_id": "VEH-001",
      "entity_type": "VEHICLE",
      "name": "Red Honda Civic ABC-123"
    },
    "claim_level": "FACT"
  },
  "ontology": {
    "allowed_relationships": ["DRIVER_OF", "OWNER_OF", "PASSENGER_IN"]
  }
}<|im_end|>
<|im_start|>assistant
"""

print("🧪 Testing model with sample validation task...\n")

In [ ]:
# Generate response
inputs = tokenizer(test_prompt, return_tensors="pt").to(model_inference.device)

with torch.no_grad():
    outputs = model_inference.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.1,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

print("📋 Model Response:")
print("="*50)
print(response)
print("="*50)

## Step 12: Download the Fine-Tuned Model

In [ ]:
# Zip the model directory
!zip -r qwen3.5-9b-qlora-finetuned.zip {output_dir}

print(f"✅ Model zipped")
print(f"\nDownload the model:")

# Download the zip file
from google.colab import files
files.download('qwen3.5-9b-qlora-finetuned.zip')

print("\n📦 Model download started!")

## Step 13: (Optional) Save to Google Drive

In [ ]:
# Mount Google Drive (if not already mounted)
from google.colab import drive
drive.mount('/content/drive')

# Copy model to Drive
!cp -r {output_dir} '/content/drive/MyDrive/qwen3.5-9b-qlora-finetuned'

print("✅ Model saved to Google Drive at: MyDrive/qwen3.5-9b-qlora-finetuned")

## 🎉 Fine-Tuning Complete!

### What you've accomplished:
- ✅ Fine-tuned Qwen 3.5 9B using QLoRA
- ✅ Reduced memory usage through 4-bit quantization
- ✅ Trained only ~1% of parameters (LoRA adapters)
- ✅ Saved the model for future use

### Next steps:
1. **Test more examples** to evaluate model performance
2. **Merge adapters** with base model for deployment (optional)
3. **Deploy** using vLLM, TGI, or Ollama
4. **Iterate** by adjusting hyperparameters if needed

### Model loading later:
```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-9B",
    device_map="auto",
    torch_dtype=torch.bfloat16
)

# Load fine-tuned adapter
model = PeftModel.from_pretrained(base_model, "./qwen3.5-9b-qlora-finetuned")
tokenizer = AutoTokenizer.from_pretrained("./qwen3.5-9b-qlora-finetuned")
```